In [1]:
#to prevent colab to automatically disconnect
import IPython
from google.colab import output

display(IPython.display.Javascript('''
 function ClickConnect(){
   btn = document.querySelector("colab-connect-button")
   if (btn != null){
     console.log("Click colab-connect-button");
     btn.click()
     }

   btn = document.getElementById('ok')
   if (btn != null){
     console.log("Click reconnect");
     btn.click()
     }
  }

setInterval(ClickConnect,60000)
'''))

print("Done.")

<IPython.core.display.Javascript object>

Done.


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# **Step 1: Importing Libraries**

In [3]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import datetime

# **Step 2:Loading Dataset**

In [4]:
crime_dataset_path = r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/1.Merging/Crime_Dataset.csv'

In [5]:
la_crime = pd.read_csv(crime_dataset_path)

In [6]:
print("Rows in Crime Dataset :",la_crime.shape[0])
print("Columns in Crime Dataset :",la_crime.shape[1])

Rows in Crime Dataset : 3158408
Columns in Crime Dataset : 4


In [7]:
la_crime.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3158408 entries, 0 to 3158407
Data columns (total 4 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   DATETIME    object
 1   AREA        int64 
 2   AREA NAME   object
 3   Crime Type  object
dtypes: int64(1), object(3)
memory usage: 96.4+ MB


In [8]:
la_crime.describe()

,AREA
count,3.158408e+06
mean,1.101049e+01
std,6.044820e+00
min,1.000000e+00
25%,6.000000e+00
50%,1.100000e+01
75%,1.600000e+01
max,2.100000e+01


# **Step 3 : Data Formatting**

**Datetime formatting**

In [9]:
la_crime["DATETIME"] = pd.to_datetime(la_crime["DATETIME"],errors ="coerce")

In [10]:
print(f"Minimum Date: {la_crime['DATETIME'].min()}")
print(f"Maximum Date: {la_crime['DATETIME'].max()}")



Minimum Date: 2010-01-01 00:01:00
Maximum Date: 2025-03-13 17:57:00


# **Step 4: Check Missing Values**

In [11]:
la_crime.isnull().sum()


,0
DATETIME,0
AREA,0
AREA NAME,0
Crime Type,0


# **Step 4: Separating Crime Dataset patrol divison wise**

In [12]:
# Sort the DataFrame in ascending order by datetime
la_crime = la_crime.sort_values(by='DATETIME')


In [13]:
patrol_divisons = la_crime[['AREA', 'AREA NAME']].drop_duplicates().set_index('AREA')['AREA NAME'].to_dict()
patrol_divisons = dict(sorted(patrol_divisons.items(), key=lambda item: item[0]))
patrol_divisons

{1: 'Central',
 2: 'Rampart',
 3: 'Southwest',
 4: 'Hollenbeck',
 5: 'Harbor',
 6: 'Hollywood',
 7: 'Wilshire',
 8: 'West LA',
 9: 'Van Nuys',
 10: 'West Valley',
 11: 'Northeast',
 12: '77th Street',
 13: 'Newton',
 14: 'Pacific',
 15: 'N Hollywood',
 16: 'Foothill',
 17: 'Devonshire',
 18: 'Southeast',
 19: 'Mission',
 20: 'Olympic',
 21: 'Topanga'}

In [14]:
#Separating Records based on the patrol divisons
patrol_wise_crime_dataset = {}
for patrol_divison in patrol_divisons.keys():
  patrol_wise_crime_dataset[int(patrol_divison)] = la_crime[la_crime["AREA"] == patrol_divison].reset_index(drop = True)

  print(f"{patrol_divisons[patrol_divison]} = {patrol_wise_crime_dataset[int(patrol_divison)].shape}")

Central = (168197, 4)
Rampart = (136773, 4)
Southwest = (193386, 4)
Hollenbeck = (115359, 4)
Harbor = (133708, 4)
Hollywood = (151787, 4)
Wilshire = (137178, 4)
West LA = (135057, 4)
Van Nuys = (142993, 4)
West Valley = (132206, 4)
Northeast = (143551, 4)
77th Street = (212860, 4)
Newton = (150891, 4)
Pacific = (172408, 4)
N Hollywood = (165745, 4)
Foothill = (120659, 4)
Devonshire = (148558, 4)
Southeast = (162745, 4)
Mission = (145517, 4)
Olympic = (147233, 4)
Topanga = (141597, 4)


In [15]:
patrol_wise_crime_dataset[13].head()

,DATETIME,AREA,AREA NAME,Crime Type
0,2010-01-01 00:01:00,13,Newton,OTHER OFFENSES
1,2010-01-01 00:01:00,13,Newton,SEX OFFENSE
2,2010-01-01 00:01:00,13,Newton,LARCENY/THEFT
3,2010-01-01 00:01:00,13,Newton,OTHER OFFENSES
4,2010-01-01 00:01:00,13,Newton,OTHER OFFENSES


In [16]:
patrol_wise_crime_dataset[1].head()

,DATETIME,AREA,AREA NAME,Crime Type
0,2010-01-01 00:01:00,1,Central,LARCENY/THEFT
1,2010-01-01 00:01:00,1,Central,ASSAULT
2,2010-01-01 00:01:00,1,Central,WEAPONS VIOLATION
3,2010-01-01 00:05:00,1,Central,LARCENY/THEFT
4,2010-01-01 00:05:00,1,Central,LARCENY/THEFT


# **Step 5 : Preprocessing**

Converting Crime Types into numerical type using to_categorical

In [17]:
# Collect all unique crime types across all patrol divisions
all_crime_types = sorted(set().union(*[patrol_wise_crime_dataset[div]["Crime Type"].unique() for div in patrol_divisons]))

# Create a mapping of crime types to their indices
crime_type_to_index = {crime: idx for idx, crime in enumerate(all_crime_types)}

crime_type_to_index

{'ASSAULT': 0,
 'BURGLARY': 1,
 'CRIMINAL TRESPASS': 2,
 'DECEPTIVE PRACTICE': 3,
 'DRUG/NARCOTIC': 4,
 'HOMICIDE': 5,
 'HUMAN TRAFFICKING': 6,
 'INTERFERENCE WITH PUBLIC OFFICER': 7,
 'KIDNAPPING': 8,
 'LARCENY/THEFT': 9,
 'OTHER OFFENSES': 10,
 'PROSTITUTION': 11,
 'PUBLIC PEACE VIOLATION': 12,
 'ROBBERY': 13,
 'SEX OFFENSE': 14,
 'WEAPONS VIOLATION': 15}

In [18]:
def count_total_crime_occurrences(datasets):
    """
    Counts the total occurrences of each crime type across multiple datasets.

    Parameters:
    datasets (dict of pd.DataFrame): A dictionary where keys are patrol divisions, and values are DataFrames.
    crime_column (str): The column name representing crime types.

    Returns:
    pd.Series: A Series with crime types as index and their total occurrences as values.
    """
    total_crime_counts = {}

    for division in datasets.keys():  # Iterate over dataset dictionary correctly

        crime_counts = datasets[division]["Crime Type"].value_counts()  # Corrected data access
        for crime, count in crime_counts.items():
            total_crime_counts[crime] = total_crime_counts.get(crime, 0) + count

    return total_crime_counts  # Sort results

# Call the function and print results
crime_counts_total = count_total_crime_occurrences(patrol_wise_crime_dataset)
crime_counts_total = dict(sorted(crime_counts_total.items()))
crime_counts_total

{'ASSAULT': 723116,
 'BURGLARY': 456030,
 'CRIMINAL TRESPASS': 41602,
 'DECEPTIVE PRACTICE': 950,
 'DRUG/NARCOTIC': 59,
 'HOMICIDE': 4370,
 'HUMAN TRAFFICKING': 1262,
 'INTERFERENCE WITH PUBLIC OFFICER': 6879,
 'KIDNAPPING': 5377,
 'LARCENY/THEFT': 1052385,
 'OTHER OFFENSES': 483374,
 'PROSTITUTION': 1093,
 'PUBLIC PEACE VIOLATION': 9632,
 'ROBBERY': 140956,
 'SEX OFFENSE': 76829,
 'WEAPONS VIOLATION': 154494}

In [19]:
for division in patrol_divisons:
    # Map Crime Type to numerical index based on the global set of crime types
    patrol_wise_crime_dataset[division]["Crime Type"] = patrol_wise_crime_dataset[division]["Crime Type"].apply(
        lambda x: crime_type_to_index.get(x, -1)  # Use -1 for unknown categories
    )

    # Remove rows with invalid crime type (-1) before encoding
    patrol_wise_crime_dataset[division] = patrol_wise_crime_dataset[division][patrol_wise_crime_dataset[division]["Crime Type"] != -1]

    # One-hot encode crime type with a fixed number of categories
    categorical = pd.DataFrame(to_categorical(patrol_wise_crime_dataset[division]["Crime Type"], num_classes=len(all_crime_types)))

    # Rename columns to match crime type names
    categorical.columns = crime_type_to_index.values()

    patrol_wise_crime_dataset[division]["Count"] = 1.0
    # Merge one-hot encoded columns into the original DataFrame
    patrol_wise_crime_dataset[division] = pd.concat([patrol_wise_crime_dataset[division], categorical], axis=1)

    # Optionally, remove original "Crime Type" column
    patrol_wise_crime_dataset[division].drop(columns=["Crime Type","AREA","AREA NAME"], inplace=True)


In [20]:
patrol_wise_crime_dataset[1].head()

,DATETIME,Count,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,2010-01-01 00:01:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2010-01-01 00:01:00,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2010-01-01 00:01:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2010-01-01 00:05:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2010-01-01 00:05:00,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0


In [21]:
patrol_wise_crime_dataset[1].shape

(168197, 18)

# **Step 6: Resampling Crime Records**

In [22]:
def aggregating_3H_crime(df):
  return df.resample("3h").sum().fillna(0)


In [23]:
for divison in patrol_divisons.keys():
  #setting datetime column as index
  patrol_wise_crime_dataset[divison].set_index("DATETIME",inplace=True)
  patrol_wise_crime_dataset[divison] = aggregating_3H_crime(patrol_wise_crime_dataset[divison])
  patrol_wise_crime_dataset[divison]["p_id"] = divison

  print(f"{patrol_divisons[divison]},{patrol_wise_crime_dataset[divison].shape}")

Central,(44170, 18)
Rampart,(44045, 18)
Southwest,(44395, 18)
Hollenbeck,(44397, 18)
Harbor,(44406, 18)
Hollywood,(43814, 18)
Wilshire,(43822, 18)
West LA,(44283, 18)
Van Nuys,(44157, 18)
West Valley,(44398, 18)
Northeast,(44277, 18)
77th Street,(44190, 18)
Newton,(44222, 18)
Pacific,(44112, 18)
N Hollywood,(44126, 18)
Foothill,(44294, 18)
Devonshire,(44398, 18)
Southeast,(44388, 18)
Mission,(44397, 18)
Olympic,(44254, 18)
Topanga,(44397, 18)


In [24]:
patrol_wise_crime_dataset[1].head()

,Count,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,p_id
DATETIME,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,11.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,0.0,0.0,0.0,1.0,1
2010-01-01 03:00:00,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1
2010-01-01 06:00:00,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,1.0,0.0,0.0,0.0,0.0,1.0,1
2010-01-01 09:00:00,17.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.0,1.0,0.0,0.0,0.0,1.0,0.0,1
2010-01-01 12:00:00,10.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,1.0,1


# **Step 7: Feature Extraction**

In [25]:
#feature extraction (creating cyclic features from datetime)
def dateTimeToSignal(df):
    '''
    Converts the DateTime index to timestamp and convert it to signal (sine and cosine) to deal with periodicity
    input:
        df : Dataset
    Output:
        df : Dataset with column Day sin , Day cos , Week sin , Week cos ,  Year sin , Year cos ; representing Sin / Cosine signal for timestamp
    '''
    timestamp_s = df.index.map(datetime.datetime.timestamp)
    day = 24*60*60
    week = 7*day
    year = (365.2425)*day

    df['Day sin'] = np.sin(timestamp_s * (2 * np.pi / day))
    df['Day cos'] = np.cos(timestamp_s * (2 * np.pi / day))

    df['Week sin'] = np.sin(timestamp_s * (2 * np.pi / week))
    df['Week cos'] = np.cos(timestamp_s * (2 * np.pi / week))

    df['Year sin'] = np.sin(timestamp_s * (2 * np.pi / year))
    df['Year cos'] = np.cos(timestamp_s * (2 * np.pi / year))
    return df

In [26]:
for divison in patrol_divisons.keys():
  patrol_wise_crime_dataset[divison] = dateTimeToSignal(patrol_wise_crime_dataset[divison])
  print(f"{patrol_divisons[divison]} : {patrol_wise_crime_dataset[divison].shape}")

Central : (44170, 24)
Rampart : (44045, 24)
Southwest : (44395, 24)
Hollenbeck : (44397, 24)
Harbor : (44406, 24)
Hollywood : (43814, 24)
Wilshire : (43822, 24)
West LA : (44283, 24)
Van Nuys : (44157, 24)
West Valley : (44398, 24)
Northeast : (44277, 24)
77th Street : (44190, 24)
Newton : (44222, 24)
Pacific : (44112, 24)
N Hollywood : (44126, 24)
Foothill : (44294, 24)
Devonshire : (44398, 24)
Southeast : (44388, 24)
Mission : (44397, 24)
Olympic : (44254, 24)
Topanga : (44397, 24)


In [27]:
patrol_wise_crime_dataset[1].head()

,Count,0,1,2,3,4,5,6,7,8,...,13,14,15,p_id,Day sin,Day cos,Week sin,Week cos,Year sin,Year cos
DATETIME,,,,,,,,,,,,,,,,,,,,,
2010-01-01 00:00:00,11.0,6.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
2010-01-01 03:00:00,3.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2010-01-01 06:00:00,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
2010-01-01 09:00:00,17.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,1,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
2010-01-01 12:00:00,10.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,1,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [28]:
patrol_wise_crime_dataset[1].columns

Index([   'Count',          0,          1,          2,          3,          4,
                5,          6,          7,          8,          9,         10,
               11,         12,         13,         14,         15,     'p_id',
        'Day sin',  'Day cos', 'Week sin', 'Week cos', 'Year sin', 'Year cos'],
      dtype='object')

# **Step 8: Grouping Certain Crime Category**

In [29]:
crime_counts_total

{'ASSAULT': 723116,
 'BURGLARY': 456030,
 'CRIMINAL TRESPASS': 41602,
 'DECEPTIVE PRACTICE': 950,
 'DRUG/NARCOTIC': 59,
 'HOMICIDE': 4370,
 'HUMAN TRAFFICKING': 1262,
 'INTERFERENCE WITH PUBLIC OFFICER': 6879,
 'KIDNAPPING': 5377,
 'LARCENY/THEFT': 1052385,
 'OTHER OFFENSES': 483374,
 'PROSTITUTION': 1093,
 'PUBLIC PEACE VIOLATION': 9632,
 'ROBBERY': 140956,
 'SEX OFFENSE': 76829,
 'WEAPONS VIOLATION': 154494}

Certain categories, such as Deceptive Practice, Drug/Narcotic, Homicide, Human Trafficking, Interference with Public Officer, and Kidnapping, have counts below 10,000. These categories are grouped into a single category, while crime types with counts exceeding 10,000 remain unchanged.

In [30]:
crime_type_to_index

{'ASSAULT': 0,
 'BURGLARY': 1,
 'CRIMINAL TRESPASS': 2,
 'DECEPTIVE PRACTICE': 3,
 'DRUG/NARCOTIC': 4,
 'HOMICIDE': 5,
 'HUMAN TRAFFICKING': 6,
 'INTERFERENCE WITH PUBLIC OFFICER': 7,
 'KIDNAPPING': 8,
 'LARCENY/THEFT': 9,
 'OTHER OFFENSES': 10,
 'PROSTITUTION': 11,
 'PUBLIC PEACE VIOLATION': 12,
 'ROBBERY': 13,
 'SEX OFFENSE': 14,
 'WEAPONS VIOLATION': 15}

In [31]:
groups = [[0],[1],[2],[9],[10],[13],[14],[15],[3,4,5,6,7,8,11,12]]

In [32]:
def grouping(dataset):
  c = 0
  for g in groups:
    if(len(g)>1):
      column_name = "Group "+str(c)
      c = c+1
      dataset[column_name] = dataset[g].sum(axis=1)
      dataset.drop(columns = g, axis=1, inplace = True)
  return  dataset


In [33]:
for divison in patrol_divisons.keys():
  patrol_wise_crime_dataset[divison] = grouping(patrol_wise_crime_dataset[divison])
  patrol_wise_crime_dataset[divison] = patrol_wise_crime_dataset[divison].rename(columns ={0:1,1:2,2:3,9:4,10:5,13:6,14:7,15:8})
  patrol_wise_crime_dataset[divison]["datetime"] = patrol_wise_crime_dataset[divison].index
  print(f"{patrol_divisons[divison]} : {patrol_wise_crime_dataset[divison].shape}")

Central : (44170, 18)
Rampart : (44045, 18)
Southwest : (44395, 18)
Hollenbeck : (44397, 18)
Harbor : (44406, 18)
Hollywood : (43814, 18)
Wilshire : (43822, 18)
West LA : (44283, 18)
Van Nuys : (44157, 18)
West Valley : (44398, 18)
Northeast : (44277, 18)
77th Street : (44190, 18)
Newton : (44222, 18)
Pacific : (44112, 18)
N Hollywood : (44126, 18)
Foothill : (44294, 18)
Devonshire : (44398, 18)
Southeast : (44388, 18)
Mission : (44397, 18)
Olympic : (44254, 18)
Topanga : (44397, 18)


In [34]:
columns_order = ["datetime","p_id",1,2,3,4,5,6,7,8,"Group 0","Count","Day sin","Day cos","Week sin","Week cos","Year sin","Year cos"]

#converting all column names to lowercase
columns_order_lower = [column.lower() if isinstance(column,str) else column for column in columns_order ]
len(columns_order_lower)

for divison in patrol_divisons.keys():
  patrol_wise_crime_dataset[divison]["datetime"] = patrol_wise_crime_dataset[divison].index
  patrol_wise_crime_dataset[divison] = patrol_wise_crime_dataset[divison].reset_index(drop = True)
  patrol_wise_crime_dataset[divison] = patrol_wise_crime_dataset[divison][columns_order]
  patrol_wise_crime_dataset[divison].columns = columns_order_lower
  print(f"{patrol_divisons[divison]} : {patrol_wise_crime_dataset[divison].shape}")

Central : (44170, 18)
Rampart : (44045, 18)
Southwest : (44395, 18)
Hollenbeck : (44397, 18)
Harbor : (44406, 18)
Hollywood : (43814, 18)
Wilshire : (43822, 18)
West LA : (44283, 18)
Van Nuys : (44157, 18)
West Valley : (44398, 18)
Northeast : (44277, 18)
77th Street : (44190, 18)
Newton : (44222, 18)
Pacific : (44112, 18)
N Hollywood : (44126, 18)
Foothill : (44294, 18)
Devonshire : (44398, 18)
Southeast : (44388, 18)
Mission : (44397, 18)
Olympic : (44254, 18)
Topanga : (44397, 18)


In [35]:
patrol_wise_crime_dataset[1].head()

,datetime,p_id,1,2,3,4,5,6,7,8,group 0,count,day sin,day cos,week sin,week cos,year sin,year cos
0,2010-01-01 00:00:00,1,6.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,0.0,11.0,-4.416858e-12,1.000000e+00,0.781831,0.623490,0.005161,0.999987
1,2010-01-01 03:00:00,1,2.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,7.071068e-01,7.071068e-01,0.846724,0.532032,0.007311,0.999973
2,2010-01-01 06:00:00,1,0.0,0.0,0.0,12.0,1.0,0.0,0.0,1.0,0.0,14.0,1.000000e+00,6.980203e-12,0.900969,0.433884,0.009461,0.999955
3,2010-01-01 09:00:00,1,2.0,0.0,0.0,13.0,1.0,0.0,1.0,0.0,0.0,17.0,7.071068e-01,-7.071068e-01,0.943883,0.330279,0.011612,0.999933
4,2010-01-01 12:00:00,1,0.0,2.0,0.0,7.0,0.0,0.0,0.0,1.0,0.0,10.0,9.543547e-12,-1.000000e+00,0.974928,0.222521,0.013762,0.999905


In [36]:

import os
def save_patrol_wise_crime(dataset, dataset_name):
    saving_path = r'/content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset'

    # Ensure correct path concatenation
    path = os.path.join(saving_path, dataset_name + ".csv")

    # Save the dataset
    dataset.to_csv(path)
    print(f"✅ Saved: {path}")

# Loop through patrol divisions and save each dataset
for division in patrol_divisons.keys():
    dataset_name = patrol_divisons[division] + f"_{division}"
    save_patrol_wise_crime(patrol_wise_crime_dataset[division], dataset_name)


✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset/Central_1.csv
✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset/Rampart_2.csv
✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset/Southwest_3.csv
✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset/Hollenbeck_4.csv
✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/Forecasting(without weather dataset)/2.Preparing Dataset to feed Model/patrol_wise_crime_dataset/Harbor_5.csv
✅ Saved: /content/drive/MyDrive/Crime Prediction/LA/Project Tasks(Codes)/For